# Landing → Bronze

Criação da estrutura no Unity Catalog, validação das extrações do ERP e ingestão das fontes na camada Bronze.

## 1. Parâmetros

In [0]:
dbutils.widgets.text("catalogo", "estoque_hospitalar", "Catálogo")
dbutils.widgets.dropdown("reprocessar", "nao", ["nao", "sim"], "Reprocessar Bronze do zero")

CATALOGO = dbutils.widgets.get("catalogo").strip()
REPROCESSAR = dbutils.widgets.get("reprocessar") == "sim"

SCHEMAS = {
    "landing": "Arquivos extraídos do ERP de estoque",
    "bronze": "Dados brutos em Delta com metadados de ingestão",
    "silver": "Dados tipados, padronizados e deduplicados",
    "gold": "Modelo dimensional, marts de BI e features de ML",
}
VOLUMES = {
    "erp": "Extrações do ERP organizadas por fonte",
    "checkpoints": "Checkpoints e schemas inferidos do Auto Loader",
}

CAMINHO_LANDING = f"/Volumes/{CATALOGO}/landing/erp"
CAMINHO_CHECKPOINTS = f"/Volumes/{CATALOGO}/landing/checkpoints/bronze"

print(f"catalogo: {CATALOGO}")
print(f"landing: {CAMINHO_LANDING}")
print(f"checkpoints: {CAMINHO_CHECKPOINTS}")

catalogo: estoque_hospitalar
landing: /Volumes/estoque_hospitalar/landing/erp
checkpoints: /Volumes/estoque_hospitalar/landing/checkpoints/bronze


## 2. Catálogo, schemas e Volumes

In [0]:
catalogos = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]

if CATALOGO not in catalogos:
    try:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO} COMMENT 'Lakehouse de gestão de estoque hospitalar'")
    except Exception as e:
        raise RuntimeError(
            f"Não foi possível criar o catálogo '{CATALOGO}'. Use o catálogo 'workspace' no widget e execute de novo."
        ) from e

spark.sql(f"USE CATALOG {CATALOGO}")

for schema, comentario in SCHEMAS.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema} COMMENT '{comentario}'")

for volume, comentario in VOLUMES.items():
    spark.sql(f"CREATE VOLUME IF NOT EXISTS landing.{volume} COMMENT '{comentario}'")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOGO}"))

databaseName
bronze
default
gold
information_schema
landing
silver


## 3. Upload dos arquivos

Upload em `<catalogo>.landing.erp`, uma pasta por fonte:

| Pasta | Arquivos |
|---|---|
| `movimento_estoque` | Parquet mensal, a partir de 24 meses |
| `nota_fiscal` | Parquet mensal, a partir de 24 meses |
| `nota_fiscal_item` | Parquet mensal, a partir de 24 meses |
| `lote_fornecedor` | Parquet |
| `saldo_estoque` | Parquet mensal, a partir de 25 meses (inclui o saldo de abertura) |
| `cadastros` | `material`, `operacao_estoque`, `local_estoque`, `fornecedor` em CSV, com ou sem data no nome |

In [0]:
# Quantidade mínima de arquivos por pasta; meses novos entram sem alterar a validação
FONTES_LANDING = {
    "movimento_estoque": 24,
    "nota_fiscal": 24,
    "nota_fiscal_item": 24,
    "lote_fornecedor": 1,
    "saldo_estoque": 25,
    "cadastros": 4,
}
CADASTROS = ["material", "operacao_estoque", "local_estoque", "fornecedor"]

for pasta in FONTES_LANDING:
    dbutils.fs.mkdirs(f"{CAMINHO_LANDING}/{pasta}")

## 4. Validação da landing

A execução é interrompida se alguma pasta tiver menos arquivos que o mínimo, arquivo vazio, mês faltando na sequência ou cadastro ausente.

In [0]:
import re
from datetime import datetime


def listar_arquivos(caminho):
    try:
        return [f for f in dbutils.fs.ls(caminho) if not f.name.endswith("/")]
    except Exception:
        return []


def meses_faltantes(nomes):
    # Extrai o AAAAMM do nome do arquivo e compara com a sequência contínua entre o primeiro e o último mês
    meses = sorted(m.group(1) for n in nomes if (m := re.search(r"_(\d{6})\.", n)))
    if len(meses) < 2:
        return []
    ini, fim = datetime.strptime(meses[0], "%Y%m"), datetime.strptime(meses[-1], "%Y%m")
    esperados, atual = [], ini
    while atual <= fim:
        esperados.append(atual.strftime("%Y%m"))
        atual = atual.replace(year=atual.year + atual.month // 12, month=atual.month % 12 + 1)
    return sorted(set(esperados) - set(meses))


def pendencias(pasta, nomes):
    if pasta == "cadastros":
        ausentes = [c for c in CADASTROS if not any(n.startswith(c) and n.endswith(".csv") for n in nomes)]
        return [f"sem {c}" for c in ausentes]
    return meses_faltantes(nomes)


inventario = []
for pasta, minimo in FONTES_LANDING.items():
    arquivos = listar_arquivos(f"{CAMINHO_LANDING}/{pasta}")
    nomes = [f.name for f in arquivos]
    inventario.append((
        pasta,
        len(arquivos),
        minimo,
        round(sum(f.size for f in arquivos) / 2**20, 1),
        sum(1 for f in arquivos if f.size == 0),
        min(nomes) if nomes else None,
        max(nomes) if nomes else None,
        ", ".join(pendencias(pasta, nomes)),
    ))

colunas = "pasta string, arquivos int, minimo int, tamanho_mb double, vazios int, primeiro string, ultimo string, pendencias string"
display(spark.createDataFrame(inventario, colunas))

problemas = [i[0] for i in inventario if i[1] < i[2] or i[4] > 0 or i[7]]
if problemas:
    raise ValueError(f"Landing incompleta ou inconsistente em: {', '.join(problemas)}")

print("[OK] Landing validada")

pasta,arquivos,minimo,tamanho_mb,vazios,primeiro,ultimo,pendencias
movimento_estoque,24,24,76.3,0,movimento_estoque_202410.parquet,movimento_estoque_202609.parquet,
nota_fiscal,24,24,0.6,0,nota_fiscal_202410.parquet,nota_fiscal_202609.parquet,
nota_fiscal_item,24,24,0.9,0,nota_fiscal_item_202410.parquet,nota_fiscal_item_202609.parquet,
lote_fornecedor,1,1,0.6,0,lote_fornecedor.parquet,lote_fornecedor.parquet,
saldo_estoque,25,25,2.0,0,saldo_estoque_202409.parquet,saldo_estoque_202609.parquet,
cadastros,4,4,0.2,0,fornecedor.csv,operacao_estoque.csv,


[OK] Landing validada


## 5. Ingestão Bronze

Auto Loader com `trigger(availableNow=True)`: processa os arquivos ainda não lidos e encerra. Cadastros em CSV são lidos como texto e reingeridos quando o arquivo é sobrescrito. Todas as tabelas recebem `_arquivo_origem`, `_dt_modificacao_arquivo` e `_dt_ingestao`.

In [0]:
from pyspark.sql import functions as F

# allowOverwrites: um cadastro reenviado com o mesmo nome é lido de novo; a Silver usa o mais recente
OPCOES_CSV = {"header": "true", "sep": ";", "encoding": "UTF-8", "cloudFiles.allowOverwrites": "true"}

# tabela, pasta, formato, opções de leitura, schema hints, filtro de arquivo
FONTES = [
    ("tb_movimento_estoque", "movimento_estoque", "parquet", {}, "cd_conta_contabil STRING", None),
    ("tb_nota_fiscal", "nota_fiscal", "parquet", {}, None, None),
    ("tb_nota_fiscal_item", "nota_fiscal_item", "parquet", {}, None, None),
    ("tb_lote_fornecedor", "lote_fornecedor", "parquet", {}, None, None),
    ("tb_saldo_estoque", "saldo_estoque", "parquet", {}, None, None),
    ("tb_material", "cadastros", "csv", OPCOES_CSV, None, "material*.csv"),
    ("tb_operacao_estoque", "cadastros", "csv", OPCOES_CSV, None, "operacao_estoque*.csv"),
    ("tb_local_estoque", "cadastros", "csv", OPCOES_CSV, None, "local_estoque*.csv"),
    ("tb_fornecedor", "cadastros", "csv", OPCOES_CSV, None, "fornecedor*.csv"),
]

if REPROCESSAR:
    for tabela, *_ in FONTES:
        spark.sql(f"DROP TABLE IF EXISTS bronze.{tabela}")
        dbutils.fs.rm(f"{CAMINHO_CHECKPOINTS}/{tabela}", True)
    print("Tabelas e checkpoints da Bronze removidos")

In [0]:
def contar(tabela):
    return spark.table(f"bronze.{tabela}").count() if spark.catalog.tableExists(f"bronze.{tabela}") else 0


def ingerir(tabela, pasta, formato, opcoes, hints, filtro):
    checkpoint = f"{CAMINHO_CHECKPOINTS}/{tabela}"
    antes = contar(tabela)

    leitor = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", formato)
        .option("cloudFiles.schemaLocation", checkpoint)
        .options(**opcoes)
    )
    # cd_conta_contabil vem como double em parte dos arquivos; o hint fixa string e o valor divergente vai para _rescued_data
    if hints:
        leitor = leitor.option("cloudFiles.schemaHints", hints)
    if filtro:
        leitor = leitor.option("pathGlobFilter", filtro)

    df = (
        leitor.load(f"{CAMINHO_LANDING}/{pasta}")
        .withColumn("_arquivo_origem", F.col("_metadata.file_path"))
        .withColumn("_dt_modificacao_arquivo", F.col("_metadata.file_modification_time"))
        .withColumn("_dt_ingestao", F.current_timestamp())
    )

    (
        df.writeStream
        .option("checkpointLocation", checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"bronze.{tabela}")
        .awaitTermination()
    )
    # Linhas novas pela diferença de contagem, já que numInputRows pode vir nulo no serverless
    return contar(tabela) - antes


for fonte in FONTES:
    print(f"{fonte[0]:<24} {ingerir(*fonte):>12,} linhas novas")

tb_movimento_estoque                0 linhas novas
tb_nota_fiscal                      0 linhas novas
tb_nota_fiscal_item                 0 linhas novas
tb_lote_fornecedor                  0 linhas novas
tb_saldo_estoque                    0 linhas novas
tb_material                         0 linhas novas
tb_operacao_estoque                 0 linhas novas
tb_local_estoque                    0 linhas novas
tb_fornecedor                       0 linhas novas


## 6. Conferência

In [0]:
display(spark.sql(" UNION ALL ".join(
    f"""SELECT '{t}' AS tabela, COUNT(*) AS linhas, COUNT(DISTINCT _arquivo_origem) AS arquivos,
               COUNT(_rescued_data) AS linhas_resgatadas, MAX(_dt_ingestao) AS ultima_ingestao
        FROM bronze.{t}"""
    for t, *_ in FONTES
)))

tabela,linhas,arquivos,linhas_resgatadas,ultima_ingestao
tb_movimento_estoque,2656470,24,693297,2026-10-20T18:59:43.905Z
tb_nota_fiscal,8932,24,0,2026-10-20T19:00:22.454Z
tb_nota_fiscal_item,24151,24,0,2026-10-20T19:00:36.461Z
tb_lote_fornecedor,27536,1,0,2026-10-20T19:00:49.169Z
tb_saldo_estoque,95519,25,0,2026-10-20T19:00:59.959Z
tb_material,871,1,0,2026-10-20T19:01:13.421Z
tb_operacao_estoque,13,1,0,2026-10-20T19:01:23.772Z
tb_local_estoque,18,1,0,2026-10-20T19:01:35.007Z
tb_fornecedor,47,1,0,2026-10-20T19:01:45.734Z
